# Budgerigar Unified M0：单一连续控制流
先实现可懂、可区分的原音流式重建。声源参数仅作为辅助探针，不再成为波形生成瓶颈。

In [ ]:
#@title 1. 更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime,json
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
 pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True);print(pull.stdout,pull.stderr)
 if pull.returncode:
  backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}');shutil.move(str(repo),str(backup));subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True);sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches();print('commit:',subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
#@title 2. Drive、冻结评估器与严格流式检查
from google.colab import drive
drive.mount('/content/drive');WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar');MANIFEST=WORK_ROOT/'manifests'/'fsdd.jsonl';EVALUATOR_CHECKPOINT=WORK_ROOT/'checkpoints'/'frozen_real_audio_digit_evaluator'/'best.pt'
import torch
if not torch.cuda.is_available():raise RuntimeError('请选择 GPU runtime')
assert EVALUATOR_CHECKPOINT.is_file(),'先在 Sensorimotor M0 notebook 完成独立评估器'
from budgerigar.unified_streaming_autoencoder import UnifiedStreamingConfig,create_unified_streaming_autoencoder
config=UnifiedStreamingConfig();model=create_unified_streaming_autoencoder(config).cuda().eval();dummy=torch.randn(2,12,config.tick_samples,device='cuda')
with torch.no_grad():full,_,_=model(dummy);state=None;parts=[]
with torch.no_grad():
 for index in range(dummy.shape[1]):value,state,_=model.stream_step(dummy[:,index],state);parts.append(value)
streamed=torch.stack(parts,1);maximum=float((full-streamed).abs().max());mean=float((full-streamed).abs().mean());parameters=sum(p.numel() for p in model.parameters());print('parameters:',parameters,'latent:',config.latent_dim,'subframe_ms:',1000*config.tick_samples/config.sample_rate/config.subframes,'stream max/mean:',maximum,mean);assert maximum<1e-4

In [ ]:
#@title 3. 因果去噪后处理微调到 1000 step
MAX_STEPS=1000 #@param {type:'integer'}
BATCH_SIZE=12 #@param {type:'integer'}
from budgerigar.train_unified_streaming import UnifiedStreamingTrainingConfig,train_unified_streaming
SOURCE_RUN=WORK_ROOT/'checkpoints'/'unified_local_phase_m0_smoke';RUN_DIR=WORK_ROOT/'checkpoints'/'unified_denoised_m0';RESUME_FROM=SOURCE_RUN/'last.pt'
training=UnifiedStreamingTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,max_train_records=1000,max_validation_records=100,content_every_steps=2,source_probe_every_steps=4,auxiliary_start_step=200,latent_contrastive_every_steps=4,frozen_content_weight=.2,boundary_weight=2.0,curvature_weight=.5)
report=train_unified_streaming(MANIFEST,RUN_DIR,EVALUATOR_CHECKPOINT,training,config,resume_from=RESUME_FROM);print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 统一表征审计与试听
best=min(report['history'],key=lambda x:x['validation_spectral_loss']+.5*(1-x['validation_frozen_output_digit_accuracy']))
m0_pass=best['validation_frozen_real_digit_accuracy']>.9 and best['validation_frozen_output_digit_accuracy']>.8 and best['validation_output_separation_ratio']>.5 and best['validation_latent_shuffled_relative_degradation']>.1 and best['validation_latent_mean_relative_degradation']>.1 and best['validation_si_sdr_db']>0
print(json.dumps(best,ensure_ascii=False,indent=2));print('m0_pass =',m0_pass)
payload=torch.load(RUN_DIR/'validation_example.pt',map_location='cpu',weights_only=False);import torchaudio
INPUT=RUN_DIR/'unified_input.wav';OUTPUT=RUN_DIR/'unified_reconstruction.wav';torchaudio.save(str(INPUT),payload['input'].unsqueeze(0),payload['sample_rate']);torchaudio.save(str(OUTPUT),payload['reconstruction'].unsqueeze(0),payload['sample_rate'])
from IPython.display import Audio,display
print('输入：');display(Audio(filename=str(INPUT)));print('重建：');display(Audio(filename=str(OUTPUT)))
if not m0_pass:print('未通过：不要接记忆或音色转换，继续按可懂度、SI-SDR、输出可分性和潜变量消融修正。')